In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1dpGNvmjPFR_bc5vl6nMXikP6lgd5lzBJ", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Task Decomposition Strategies — Hands-On Lab

Welcome to your fourth hands-on lab for Domain 1 of the Claude Certified Architect exam. In this notebook, we will **build both fixed and dynamic task decomposition systems from scratch** — comparing prompt chaining pipelines with adaptive, model-driven decomposition strategies.

By the end of this notebook, you will be able to:
- Implement a fixed sequential pipeline (prompt chaining) for well-defined workflows
- Build a dynamic adaptive decomposition engine that plans based on discovered context
- Apply the per-file analysis + cross-file integration pass pattern
- Choose between fixed and dynamic strategies based on task characteristics

**Estimated time: 50–60 minutes**

## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/agentic-architecture/practice/4/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you receive two very different requests:

**Request A:** "Process this loan application" — you know exactly what needs to happen: verify identity, check credit score, calculate terms, generate an offer letter. Every step is known in advance.

**Request B:** "There is a bug somewhere in this codebase causing intermittent auth failures" — you have no idea how many files are involved, what the root cause is, or how many steps it will take to find and fix it.

These two requests demand fundamentally different decomposition strategies. Request A is a **fixed sequential pipeline** — predetermined steps executed in order. Request B requires **dynamic adaptive decomposition** — the agent must explore, assess, and adapt its plan as it discovers new information.

The exam tests your judgment on when to use each approach. Getting this wrong means either over-engineering simple workflows or under-equipping agents for complex, exploratory tasks. Let us build both and develop the intuition to choose correctly.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — Pipelines vs Adaptive Plans

Think of it this way. A fixed pipeline is like an assembly line in a factory — each station does its job, passes the product to the next station, and the output is predictable. An adaptive decomposition is like a detective investigating a case — you start with a crime scene, follow leads, and each discovery changes where you look next.

In [ ]:
# Setup
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Callable
from datetime import datetime, timezone
import textwrap

# Assembly line: every step is known
LOAN_PIPELINE = [
    "Step 1: Verify applicant identity",
    "Step 2: Pull credit score",
    "Step 3: Calculate loan terms",
    "Step 4: Generate offer letter",
]

# Detective work: steps emerge from discoveries
BUG_INVESTIGATION = [
    "Step 1: Map the codebase → discover 47 files, 8 modules",
    "Step 2: (Agent decides) Focus on the 3 auth-related modules",
    "Step 3: (Agent decides) Analyze auth/login.py → find suspicious retry logic",
    "Step 4: (Agent decides) Check auth/token.py → confirm stale token handling",
    "Step 5: (Agent decides) Write and verify a fix",
]

print("FIXED PIPELINE (Loan Application):")
for step in LOAN_PIPELINE:
    print(f"  {step}")
print(f"  Total steps: always {len(LOAN_PIPELINE)}")
print()
print("DYNAMIC DECOMPOSITION (Bug Investigation):")
for step in BUG_INVESTIGATION:
    print(f"  {step}")
print(f"  Total steps: unknown until runtime (was {len(BUG_INVESTIGATION)} this time)")

In [ ]:
#@title 🎧 Listen: Pipeline Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_pipeline_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Core Implementation — Fixed Sequential Pipeline (Prompt Chaining)

In [ ]:
@dataclass
class PipelineStep:
    """A single step in a fixed sequential pipeline."""
    name: str
    prompt_template: str
    tools: List[str] = field(default_factory=list)
    required_inputs: List[str] = field(default_factory=list)
    output_key: str = ""

@dataclass
class PipelineResult:
    """Result from a single pipeline step."""
    step_name: str
    output: Dict[str, Any]
    success: bool
    duration_ms: int = 0

class PromptChainPipeline:
    """Fixed sequential pipeline — each step feeds into the next."""

    def __init__(self, name: str, steps: List[PipelineStep]):
        self.name = name
        self.steps = steps
        self.results: List[PipelineResult] = []
        self.context: Dict[str, Any] = {}

    def run(self, initial_input: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
        """Execute all steps in sequence, passing outputs forward."""
        self.context = initial_input.copy()
        self.results = []

        if verbose:
            print(f"{'='*60}")
            print(f"Pipeline: {self.name}")
            print(f"Steps: {len(self.steps)}")
            print(f"Input: {json.dumps(initial_input, indent=2)}")
            print(f"{'='*60}")

        for i, step in enumerate(self.steps, 1):
            if verbose:
                print(f"\n--- Step {i}/{len(self.steps)}: {step.name} ---")

            # Check required inputs
            missing = [r for r in step.required_inputs if r not in self.context]
            if missing:
                if verbose:
                    print(f"  FAILED: Missing required inputs: {missing}")
                self.results.append(PipelineResult(
                    step_name=step.name,
                    output={"error": f"Missing inputs: {missing}"},
                    success=False
                ))
                return {"error": f"Pipeline failed at step {i}", "results": self.results}

            # Build prompt from template and context
            prompt = step.prompt_template.format(**self.context)
            if verbose:
                print(f"  Prompt: {prompt[:100]}...")

            # Execute step (mock — in real system, this calls the model)
            output = self._execute_step(step, prompt)
            if verbose:
                print(f"  Output: {json.dumps(output)}")

            # Store result and update context
            self.results.append(PipelineResult(
                step_name=step.name, output=output, success=True
            ))
            if step.output_key:
                self.context[step.output_key] = output

        if verbose:
            print(f"\n{'='*60}")
            print(f"Pipeline complete! {len(self.results)} steps executed.")
            print(f"{'='*60}")

        return self.context

    def _execute_step(self, step: PipelineStep, prompt: str) -> Dict[str, Any]:
        """Mock step execution — simulates model + tool calls."""
        # Simulated outputs based on step name
        mock_outputs = {
            "verify_identity": {"verified": True, "method": "2FA", "confidence": 0.99},
            "pull_credit": {"score": 742, "rating": "good", "history_years": 8},
            "calculate_terms": {"apr": 5.25, "term_months": 36, "monthly_payment": 452.17},
            "generate_offer": {"offer_id": f"OFFER-{uuid.uuid4().hex[:6]}",
                              "status": "ready", "expires": "2024-04-15"},
            "extract_entities": {"entities": ["machine learning", "transformer", "attention mechanism"]},
            "research_entities": {"findings": {"machine learning": "broad field",
                                               "transformer": "Vaswani et al. 2017",
                                               "attention mechanism": "key innovation"}},
            "write_summary": {"summary": "A comprehensive overview of transformer architecture..."},
        }
        return mock_outputs.get(step.name, {"result": "completed"})

print("PromptChainPipeline class defined!")

In [ ]:
#@title 🎧 Listen: Loan Pipeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_loan_pipeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Build and run a loan application pipeline

loan_pipeline = PromptChainPipeline(
    name="Loan Application Processing",
    steps=[
        PipelineStep(
            name="verify_identity",
            prompt_template="Verify the identity of applicant {applicant_name} (ID: {applicant_id}). Use 2FA verification.",
            tools=["verify_2fa", "check_identity_db"],
            required_inputs=["applicant_name", "applicant_id"],
            output_key="identity"
        ),
        PipelineStep(
            name="pull_credit",
            prompt_template="Pull credit report for verified applicant {applicant_name}. Identity verification: {identity}",
            tools=["credit_bureau_api"],
            required_inputs=["applicant_name", "identity"],
            output_key="credit"
        ),
        PipelineStep(
            name="calculate_terms",
            prompt_template="Calculate loan terms for {loan_amount} based on credit score {credit}.",
            tools=["rate_calculator"],
            required_inputs=["loan_amount", "credit"],
            output_key="terms"
        ),
        PipelineStep(
            name="generate_offer",
            prompt_template="Generate offer letter for {applicant_name}, amount {loan_amount}, terms: {terms}.",
            tools=["document_generator"],
            required_inputs=["applicant_name", "loan_amount", "terms"],
            output_key="offer"
        ),
    ]
)

result = loan_pipeline.run({
    "applicant_name": "Alice Smith",
    "applicant_id": "SSN-XXX-XX-1234",
    "loan_amount": 15000,
})

Notice how each step's output becomes the next step's input through the shared `context` dictionary. The pipeline is completely predictable — you always know what step 3 will do because steps 1 and 2 are fixed. This is ideal for well-defined workflows where regulatory requirements dictate the exact sequence.

In [ ]:
#@title 🎧 Listen: Dynamic Engine
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_dynamic_engine.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Dynamic Adaptive Decomposition

Now let us build the opposite approach — an agent that decides what to do next based on what it discovers.

In [ ]:
@dataclass
class Discovery:
    """Something the agent discovered during exploration."""
    source: str
    finding: str
    relevance: float  # 0.0 to 1.0
    follow_up_actions: List[str] = field(default_factory=list)

@dataclass
class AdaptivePlan:
    """A dynamically created plan based on discoveries."""
    steps: List[Dict[str, Any]] = field(default_factory=list)
    priority_order: List[int] = field(default_factory=list)
    rationale: str = ""

class DynamicDecompositionEngine:
    """Agent that maps territory, assesses findings, and creates prioritized plans."""

    def __init__(self):
        self.discoveries: List[Discovery] = []
        self.plan: Optional[AdaptivePlan] = None
        self.execution_log: List[Dict] = []
        self.step_count = 0

    def map_structure(self, target: Dict[str, Any], verbose: bool = True) -> List[Discovery]:
        """Phase 1: Map the territory — discover what we are working with."""
        self.step_count += 1
        if verbose:
            print(f"\n[Step {self.step_count}] MAPPING STRUCTURE")
            print(f"  Target: {target.get('name', 'unknown')}")

        discoveries = []
        for item_name, item_data in target.get("items", {}).items():
            relevance = item_data.get("relevance", 0.5)
            finding = item_data.get("description", "No description")
            follow_ups = item_data.get("follow_ups", [])

            d = Discovery(
                source=item_name,
                finding=finding,
                relevance=relevance,
                follow_up_actions=follow_ups,
            )
            discoveries.append(d)
            if verbose:
                print(f"  Discovered: {item_name} (relevance: {relevance:.1f})")

        self.discoveries.extend(discoveries)
        self.execution_log.append({
            "step": self.step_count, "action": "map_structure",
            "items_found": len(discoveries)
        })

        return discoveries

    def assess_and_prioritize(self, threshold: float = 0.6, verbose: bool = True) -> AdaptivePlan:
        """Phase 2: Assess discoveries and create a prioritized plan."""
        self.step_count += 1
        if verbose:
            print(f"\n[Step {self.step_count}] ASSESSING & PRIORITIZING")

        # Filter to relevant discoveries
        relevant = [d for d in self.discoveries if d.relevance >= threshold]
        relevant.sort(key=lambda d: d.relevance, reverse=True)

        if verbose:
            print(f"  Total discoveries: {len(self.discoveries)}")
            print(f"  Above threshold ({threshold}): {len(relevant)}")

        # Build prioritized plan from relevant discoveries
        steps = []
        for i, d in enumerate(relevant):
            for action in d.follow_up_actions:
                steps.append({
                    "priority": i + 1,
                    "source": d.source,
                    "action": action,
                    "reason": f"Relevance {d.relevance:.1f}: {d.finding}",
                })

        self.plan = AdaptivePlan(
            steps=steps,
            priority_order=list(range(len(steps))),
            rationale=f"Focused on {len(relevant)} high-relevance items out of {len(self.discoveries)} total",
        )

        if verbose:
            print(f"  Plan created: {len(steps)} actions")
            for s in steps:
                print(f"    [{s['priority']}] {s['action']} (from {s['source']})")

        self.execution_log.append({
            "step": self.step_count, "action": "assess_and_prioritize",
            "plan_size": len(steps)
        })

        return self.plan

    def execute_plan(self, verbose: bool = True) -> List[Dict]:
        """Phase 3: Execute the plan, adapting if new information emerges."""
        if not self.plan:
            return [{"error": "No plan created. Run assess_and_prioritize first."}]

        results = []
        for step_info in self.plan.steps:
            self.step_count += 1
            if verbose:
                print(f"\n[Step {self.step_count}] EXECUTING: {step_info['action']}")
                print(f"  Source: {step_info['source']}")
                print(f"  Reason: {step_info['reason']}")

            # Simulated execution — in real system, this runs tools
            result = self._simulate_execution(step_info)
            results.append(result)

            if verbose:
                print(f"  Result: {result.get('finding', 'completed')}")

            # Check for adaptive re-planning
            if result.get("needs_replan"):
                if verbose:
                    print(f"  ** NEW INFORMATION — re-planning required **")
                self.discoveries.append(Discovery(
                    source=step_info["source"],
                    finding=result["finding"],
                    relevance=result.get("relevance", 0.9),
                    follow_up_actions=result.get("new_actions", []),
                ))

            self.execution_log.append({
                "step": self.step_count, "action": step_info["action"],
                "result": result.get("finding", "completed")
            })

        return results

    def _simulate_execution(self, step_info: Dict) -> Dict:
        """Simulate executing a plan step."""
        simulated_results = {
            "analyze auth/login.py": {
                "finding": "Found retry logic with no backoff — could cause token exhaustion",
                "severity": "high",
                "needs_replan": True,
                "relevance": 0.95,
                "new_actions": ["check auth/token.py for stale token handling"],
            },
            "analyze auth/session.py": {
                "finding": "Session management looks correct, TTL properly configured",
                "severity": "low",
            },
            "analyze auth/token.py": {
                "finding": "Token refresh uses cached value without checking expiry — root cause found",
                "severity": "critical",
            },
            "check auth/token.py for stale token handling": {
                "finding": "Confirmed: stale tokens served from cache, expiry check missing",
                "severity": "critical",
            },
        }
        return simulated_results.get(step_info["action"],
                                      {"finding": f"Analyzed {step_info['source']}", "severity": "none"})

print("DynamicDecompositionEngine defined!")

In [ ]:
#@title 🎧 Listen: Dynamic Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_dynamic_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Run a dynamic decomposition on a codebase bug investigation

engine = DynamicDecompositionEngine()

# Phase 1: Map the codebase
codebase = {
    "name": "auth-service",
    "items": {
        "auth/login.py": {
            "description": "Handles user login and 2FA",
            "relevance": 0.9,
            "follow_ups": ["analyze auth/login.py"],
        },
        "auth/session.py": {
            "description": "Session management and TTL",
            "relevance": 0.7,
            "follow_ups": ["analyze auth/session.py"],
        },
        "auth/token.py": {
            "description": "Token generation and refresh",
            "relevance": 0.85,
            "follow_ups": ["analyze auth/token.py"],
        },
        "auth/utils.py": {
            "description": "Utility functions for hashing",
            "relevance": 0.3,
            "follow_ups": ["analyze auth/utils.py"],
        },
        "auth/middleware.py": {
            "description": "Request authentication middleware",
            "relevance": 0.5,
            "follow_ups": ["analyze auth/middleware.py"],
        },
    }
}

discoveries = engine.map_structure(codebase)

# Phase 2: Assess and prioritize — only focus on high-relevance files
plan = engine.assess_and_prioritize(threshold=0.6)

# Phase 3: Execute the plan
results = engine.execute_plan()

# Summary
print(f"\n{'='*60}")
print(f"INVESTIGATION SUMMARY")
print(f"{'='*60}")
print(f"Total steps taken: {engine.step_count}")
print(f"Files mapped: {len(codebase['items'])}")
print(f"Files analyzed: {len(plan.steps)}")
print(f"Files skipped (low relevance): {len(codebase['items']) - len(plan.steps)}")
print(f"\nPlan rationale: {plan.rationale}")

The dynamic engine mapped 5 files, decided that only 3 were relevant enough to analyze, and then — critically — when analyzing `login.py` discovered new information that triggered a re-plan. This adaptive behavior is impossible with a fixed pipeline.

In [ ]:
#@title 🎧 Listen: Per File Cross File
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_per_file_cross_file.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Per-File Analysis + Cross-File Integration Pass

This is a particularly powerful pattern the exam covers. First, analyze each file independently. Then, in a separate integration pass, look for cross-cutting patterns.

In [ ]:
class PerFileCrossFileAnalyzer:
    """Two-phase pattern: per-file analysis followed by cross-file integration."""

    def __init__(self):
        self.per_file_results: Dict[str, Dict] = {}
        self.cross_file_findings: List[Dict] = []

    def analyze_file(self, filename: str, content: Dict, verbose: bool = True) -> Dict:
        """Phase 1: Analyze a single file in isolation."""
        if verbose:
            print(f"  Analyzing: {filename}")

        # Simulated per-file analysis
        analysis = {
            "filename": filename,
            "imports": content.get("imports", []),
            "functions": content.get("functions", []),
            "patterns": content.get("patterns", []),
            "issues": content.get("issues", []),
            "dependencies": content.get("dependencies", []),
        }

        self.per_file_results[filename] = analysis
        if verbose:
            print(f"    Functions: {analysis['functions']}")
            print(f"    Patterns: {analysis['patterns']}")
            if analysis['issues']:
                print(f"    Issues: {analysis['issues']}")

        return analysis

    def run_per_file_phase(self, files: Dict[str, Dict], verbose: bool = True):
        """Run Phase 1 on all files."""
        if verbose:
            print(f"{'='*60}")
            print(f"PHASE 1: Per-File Analysis ({len(files)} files)")
            print(f"{'='*60}")

        for filename, content in files.items():
            self.analyze_file(filename, content, verbose)

    def run_cross_file_phase(self, verbose: bool = True) -> List[Dict]:
        """Phase 2: Look for cross-cutting patterns across all per-file results."""
        if verbose:
            print(f"\n{'='*60}")
            print(f"PHASE 2: Cross-File Integration")
            print(f"{'='*60}")

        self.cross_file_findings = []

        # Check 1: Circular dependencies
        all_deps = {}
        for fname, analysis in self.per_file_results.items():
            all_deps[fname] = analysis.get("dependencies", [])

        for fname, deps in all_deps.items():
            for dep in deps:
                if dep in all_deps and fname in all_deps.get(dep, []):
                    finding = {
                        "type": "circular_dependency",
                        "files": [fname, dep],
                        "severity": "high",
                        "description": f"Circular dependency between {fname} and {dep}",
                    }
                    # Avoid duplicate findings
                    existing = [f for f in self.cross_file_findings
                                if set(f.get("files", [])) == set(finding["files"])]
                    if not existing:
                        self.cross_file_findings.append(finding)

        # Check 2: Shared patterns across files
        pattern_files = {}
        for fname, analysis in self.per_file_results.items():
            for pattern in analysis.get("patterns", []):
                if pattern not in pattern_files:
                    pattern_files[pattern] = []
                pattern_files[pattern].append(fname)

        for pattern, files_with in pattern_files.items():
            if len(files_with) > 1:
                self.cross_file_findings.append({
                    "type": "shared_pattern",
                    "files": files_with,
                    "severity": "info",
                    "description": f"Pattern '{pattern}' found in {len(files_with)} files: {files_with}",
                })

        # Check 3: Issues that span multiple files
        all_issues = {}
        for fname, analysis in self.per_file_results.items():
            for issue in analysis.get("issues", []):
                category = issue.split(":")[0] if ":" in issue else issue
                if category not in all_issues:
                    all_issues[category] = []
                all_issues[category].append({"file": fname, "issue": issue})

        for category, occurrences in all_issues.items():
            if len(occurrences) > 1:
                self.cross_file_findings.append({
                    "type": "systemic_issue",
                    "files": [o["file"] for o in occurrences],
                    "severity": "high",
                    "description": f"Systemic issue '{category}' found in {len(occurrences)} files",
                    "details": occurrences,
                })

        if verbose:
            print(f"  Cross-file findings: {len(self.cross_file_findings)}")
            for f in self.cross_file_findings:
                print(f"    [{f['severity'].upper()}] {f['description']}")

        return self.cross_file_findings

print("PerFileCrossFileAnalyzer defined!")

In [ ]:
# Run the two-phase analyzer on a simulated codebase

analyzer = PerFileCrossFileAnalyzer()

files = {
    "auth/login.py": {
        "imports": ["auth.token", "auth.session"],
        "functions": ["login", "verify_2fa", "handle_login_error"],
        "patterns": ["retry_without_backoff", "error_swallowing"],
        "issues": ["error_handling: catches all exceptions silently",
                    "retry: no exponential backoff on failures"],
        "dependencies": ["auth/token.py", "auth/session.py"],
    },
    "auth/token.py": {
        "imports": ["auth.login", "jwt"],
        "functions": ["generate_token", "refresh_token", "validate_token"],
        "patterns": ["retry_without_backoff", "cache_without_ttl"],
        "issues": ["error_handling: no fallback when token refresh fails",
                    "caching: cached tokens have no TTL check"],
        "dependencies": ["auth/login.py"],
    },
    "auth/session.py": {
        "imports": ["redis", "auth.token"],
        "functions": ["create_session", "get_session", "expire_session"],
        "patterns": ["proper_ttl"],
        "issues": [],
        "dependencies": ["auth/token.py"],
    },
    "api/routes.py": {
        "imports": ["auth.login", "auth.session"],
        "functions": ["handle_request", "validate_input", "send_response"],
        "patterns": ["error_swallowing"],
        "issues": ["error_handling: returns 200 OK on internal errors"],
        "dependencies": ["auth/login.py", "auth/session.py"],
    },
}

# Phase 1: Analyze each file independently
analyzer.run_per_file_phase(files)

# Phase 2: Cross-file integration
findings = analyzer.run_cross_file_phase()

The per-file analysis found issues within each file. But the cross-file integration pass discovered patterns that no single-file analysis could find: circular dependencies between `login.py` and `token.py`, the `retry_without_backoff` pattern appearing in multiple files (suggesting a systemic practice rather than a one-off mistake), and `error_handling` issues spanning 3 files (suggesting a team-wide problem). This two-phase approach is exactly what the exam expects you to understand.

In [ ]:
#@title 🎧 Listen: Exercise1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_exercise1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Exercise 1 — Build a Document Processing Pipeline

In [ ]:
# ============ TODO ============
# Build a fixed sequential pipeline for document processing:
#   Step 1: "extract_entities" — extract key entities from the document
#     - required_inputs: ["document_text"]
#     - output_key: "entities"
#   Step 2: "research_entities" — research each entity
#     - required_inputs: ["entities"]
#     - output_key: "research"
#   Step 3: "write_summary" — write a summary incorporating research
#     - required_inputs: ["document_text", "entities", "research"]
#     - output_key: "summary"
#
# Use the PromptChainPipeline class defined above.
# Run it with initial_input: {"document_text": "The transformer architecture..."}

# YOUR CODE HERE
# doc_pipeline = PromptChainPipeline(...)
# result = doc_pipeline.run(...)
# ==============================

### Exercise 1 — Solution

In [ ]:
doc_pipeline = PromptChainPipeline(
    name="Document Processing Pipeline",
    steps=[
        PipelineStep(
            name="extract_entities",
            prompt_template="Extract key entities from this document: {document_text}",
            tools=["entity_extractor"],
            required_inputs=["document_text"],
            output_key="entities",
        ),
        PipelineStep(
            name="research_entities",
            prompt_template="Research the following entities: {entities}",
            tools=["knowledge_base_search"],
            required_inputs=["entities"],
            output_key="research",
        ),
        PipelineStep(
            name="write_summary",
            prompt_template="Write a summary of '{document_text}' using entities {entities} and research {research}.",
            tools=["text_generator"],
            required_inputs=["document_text", "entities", "research"],
            output_key="summary",
        ),
    ]
)

result = doc_pipeline.run({
    "document_text": "The transformer architecture revolutionized NLP with self-attention mechanisms..."
})

print(f"\nFinal summary: {result.get('summary', {}).get('summary', 'N/A')}")

In [ ]:
#@title 🎧 Listen: Exercise2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_exercise2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 2 — Implement a Decision Framework

In [ ]:
# ============ TODO ============
# Build a function `choose_strategy(task)` that analyzes a task description
# and returns whether to use "fixed_pipeline" or "dynamic_decomposition".
#
# Rules:
#   - If task has a "steps" field with a list → "fixed_pipeline"
#   - If task has "exploratory": True → "dynamic_decomposition"
#   - If task has "step_count": "unknown" → "dynamic_decomposition"
#   - If task has "regulatory": True → "fixed_pipeline" (compliance needs predictability)
#   - Default: "fixed_pipeline"
#
# Also return a brief "rationale" string explaining why.
#
# Example:
#   choose_strategy({"name": "Loan Application", "steps": [...], "regulatory": True})
#   → {"strategy": "fixed_pipeline", "rationale": "Known steps + regulatory requirement"}

def choose_strategy(task: Dict) -> Dict:
    # YOUR CODE HERE
    pass

# Test cases:
# tasks = [
#     {"name": "Loan Processing", "steps": ["verify", "credit", "terms", "offer"], "regulatory": True},
#     {"name": "Bug Investigation", "exploratory": True, "step_count": "unknown"},
#     {"name": "Data Migration", "steps": ["extract", "transform", "load"]},
#     {"name": "Code Review", "exploratory": True, "step_count": "unknown"},
#     {"name": "Invoice Generation", "steps": ["gather", "calculate", "format", "send"], "regulatory": True},
# ]
# for task in tasks:
#     result = choose_strategy(task)
#     print(f"  {task['name']}: {result['strategy']} — {result['rationale']}")
# ==============================

### Exercise 2 — Solution

In [ ]:
def choose_strategy(task: Dict) -> Dict:
    """Decide between fixed pipeline and dynamic decomposition."""
    reasons = []

    has_steps = "steps" in task and isinstance(task["steps"], list)
    is_exploratory = task.get("exploratory", False)
    unknown_count = task.get("step_count") == "unknown"
    is_regulatory = task.get("regulatory", False)

    # Dynamic signals
    if is_exploratory:
        reasons.append("exploratory task")
    if unknown_count:
        reasons.append("unknown number of steps")

    # Fixed signals
    if has_steps:
        reasons.append(f"{len(task['steps'])} known steps")
    if is_regulatory:
        reasons.append("regulatory compliance needs predictability")

    # Decision logic
    if is_regulatory and has_steps:
        strategy = "fixed_pipeline"
    elif is_exploratory or unknown_count:
        strategy = "dynamic_decomposition"
    elif has_steps:
        strategy = "fixed_pipeline"
    else:
        strategy = "fixed_pipeline"

    return {
        "strategy": strategy,
        "rationale": " + ".join(reasons) if reasons else "default to fixed pipeline"
    }

# Test
tasks = [
    {"name": "Loan Processing", "steps": ["verify", "credit", "terms", "offer"], "regulatory": True},
    {"name": "Bug Investigation", "exploratory": True, "step_count": "unknown"},
    {"name": "Data Migration", "steps": ["extract", "transform", "load"]},
    {"name": "Code Review", "exploratory": True, "step_count": "unknown"},
    {"name": "Invoice Generation", "steps": ["gather", "calculate", "format", "send"], "regulatory": True},
]

print("Strategy Selection Results:")
print(f"{'Task':<22} {'Strategy':<25} {'Rationale'}")
print("=" * 80)
for task in tasks:
    result = choose_strategy(task)
    print(f"  {task['name']:<20} {result['strategy']:<25} {result['rationale']}")

In [ ]:
#@title 🎧 Listen: Exercise3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_exercise3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 3 — Per-File to Cross-File Pattern

In [ ]:
# ============ TODO ============
# Use the PerFileCrossFileAnalyzer to analyze a NEW set of files.
# Your task: create a files dict with at least 3 files that will produce:
#   1. At least one circular dependency (file A depends on file B, B depends on A)
#   2. At least one shared pattern across 2+ files
#   3. At least one systemic issue across 2+ files
#
# Then run both phases and verify all 3 cross-file findings appear.

# YOUR CODE HERE
# my_files = { ... }
# analyzer2 = PerFileCrossFileAnalyzer()
# analyzer2.run_per_file_phase(my_files)
# findings = analyzer2.run_cross_file_phase()
# assert len([f for f in findings if f["type"] == "circular_dependency"]) >= 1
# assert len([f for f in findings if f["type"] == "shared_pattern"]) >= 1
# assert len([f for f in findings if f["type"] == "systemic_issue"]) >= 1
# print("All 3 cross-file patterns detected!")
# ==============================

### Exercise 3 — Solution

In [ ]:
my_files = {
    "services/payment.py": {
        "imports": ["services.inventory"],
        "functions": ["charge_card", "process_payment", "refund"],
        "patterns": ["no_input_validation", "hardcoded_config"],
        "issues": ["security: no input sanitization on payment amounts",
                    "logging: sensitive card data written to logs"],
        "dependencies": ["services/inventory.py"],
    },
    "services/inventory.py": {
        "imports": ["services.payment"],
        "functions": ["check_stock", "reserve_item", "release_item"],
        "patterns": ["no_input_validation", "race_condition"],
        "issues": ["security: no bounds check on quantity parameter",
                    "concurrency: stock updates not atomic"],
        "dependencies": ["services/payment.py"],
    },
    "services/shipping.py": {
        "imports": ["services.inventory"],
        "functions": ["create_shipment", "track_shipment", "cancel_shipment"],
        "patterns": ["hardcoded_config"],
        "issues": ["logging: API keys included in debug logs"],
        "dependencies": ["services/inventory.py"],
    },
}

analyzer2 = PerFileCrossFileAnalyzer()
analyzer2.run_per_file_phase(my_files)
findings = analyzer2.run_cross_file_phase()

# Verify
circ = [f for f in findings if f["type"] == "circular_dependency"]
shared = [f for f in findings if f["type"] == "shared_pattern"]
systemic = [f for f in findings if f["type"] == "systemic_issue"]

print(f"\nVerification:")
print(f"  Circular dependencies: {len(circ)} (need >= 1)")
print(f"  Shared patterns: {len(shared)} (need >= 1)")
print(f"  Systemic issues: {len(systemic)} (need >= 1)")

assert len(circ) >= 1, "Missing circular dependency!"
assert len(shared) >= 1, "Missing shared pattern!"
assert len(systemic) >= 1, "Missing systemic issue!"
print("\nAll 3 cross-file patterns detected!")

In [ ]:
#@title 🎧 Listen: Exercise4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_exercise4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 4 — Adaptive Re-Planning

In [ ]:
# ============ TODO ============
# Extend the DynamicDecompositionEngine to support re-planning.
# When execute_plan encounters a result with "needs_replan": True,
# it should:
#   1. Add the new discovery to self.discoveries
#   2. Run assess_and_prioritize again with the NEW discoveries
#   3. Continue executing the NEW plan
#
# Write a function `run_with_replanning(engine, target)` that:
#   a) Calls engine.map_structure(target)
#   b) Calls engine.assess_and_prioritize(threshold=0.6)
#   c) Iterates through plan steps one at a time
#   d) If a step result has "needs_replan", re-runs assess_and_prioritize
#      and continues with the new plan
#   e) Returns the total number of steps actually executed

def run_with_replanning(engine, target):
    # YOUR CODE HERE
    pass

# Test:
# engine2 = DynamicDecompositionEngine()
# total_steps = run_with_replanning(engine2, codebase)  # codebase defined earlier
# print(f"Total steps executed: {total_steps}")
# ==============================

### Exercise 4 — Solution

In [ ]:
def run_with_replanning(engine, target, verbose=True):
    """Run dynamic decomposition with adaptive re-planning."""
    # Phase 1: Map
    engine.map_structure(target, verbose=verbose)

    # Phase 2: Initial plan
    plan = engine.assess_and_prioritize(threshold=0.6, verbose=verbose)

    total_executed = 0
    executed_actions = set()
    max_replans = 3
    replan_count = 0

    while plan.steps and replan_count <= max_replans:
        for step_info in plan.steps:
            action = step_info["action"]

            # Skip already-executed actions
            if action in executed_actions:
                continue

            engine.step_count += 1
            total_executed += 1
            executed_actions.add(action)

            if verbose:
                print(f"\n[Step {engine.step_count}] EXECUTING: {action}")
                print(f"  Source: {step_info['source']}")

            result = engine._simulate_execution(step_info)

            if verbose:
                print(f"  Result: {result.get('finding', 'completed')}")

            # Check for re-planning trigger
            if result.get("needs_replan"):
                replan_count += 1
                if verbose:
                    print(f"  ** RE-PLANNING (#{replan_count}) — new information found **")

                engine.discoveries.append(Discovery(
                    source=step_info["source"],
                    finding=result["finding"],
                    relevance=result.get("relevance", 0.9),
                    follow_up_actions=result.get("new_actions", []),
                ))

                plan = engine.assess_and_prioritize(threshold=0.6, verbose=verbose)
                break  # Restart with new plan
        else:
            break  # All steps executed without re-plan

    if verbose:
        print(f"\n{'='*60}")
        print(f"COMPLETE: {total_executed} steps executed, {replan_count} re-plans")

    return total_executed

# Test with the codebase from earlier
engine2 = DynamicDecompositionEngine()
total = run_with_replanning(engine2, codebase)

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Visualization — Strategy Comparison

In [ ]:
def visualize_strategy_comparison():
    """Visualize the key differences between fixed and dynamic strategies."""
    print("=" * 70)
    print("          TASK DECOMPOSITION STRATEGY COMPARISON")
    print("=" * 70)
    print()

    # Fixed Pipeline
    print("  FIXED SEQUENTIAL PIPELINE (Prompt Chaining)")
    print("  " + "-" * 50)
    print()
    steps = ["Extract", "Research", "Calculate", "Generate"]
    for i, step in enumerate(steps):
        if i > 0:
            print("       │")
            print("       ▼")
        print(f"  ┌─────────────────────────┐")
        print(f"  │  Step {i+1}: {step:<15} │")
        print(f"  └─────────────────────────┘")
    print()
    print("  Properties: Predictable | Debuggable | Deterministic")
    print("  Best for:   Known workflows, regulatory compliance")
    print()

    # Dynamic Decomposition
    print("  DYNAMIC ADAPTIVE DECOMPOSITION")
    print("  " + "-" * 50)
    print()
    print("  ┌──────────────┐")
    print("  │  Map Target   │")
    print("  └──────┬───────┘")
    print("         │")
    print("         ▼")
    print("  ┌──────────────┐     ┌────────────────┐")
    print("  │  Assess &    │────>│  File A (0.9)   │")
    print("  │  Prioritize  │     │  File B (0.85)  │")
    print("  └──────┬───────┘     │  File C (0.7)   │")
    print("         │             │  File D (0.3) x │ <- skipped")
    print("         ▼             └────────────────┘")
    print("  ┌──────────────┐")
    print("  │  Execute     │──── new info? ──── re-plan!")
    print("  │  (adaptive)  │")
    print("  └──────────────┘")
    print()
    print("  Properties: Flexible | Model-driven | Exploratory")
    print("  Best for:   Bug investigation, research, unknown scope")
    print()

    # Decision framework
    print("  DECISION FRAMEWORK")
    print("  " + "-" * 50)
    print(f"  {'Signal':<35} {'Strategy'}")
    print(f"  {'='*35} {'='*25}")
    signals = [
        ("Steps known in advance", "Fixed Pipeline"),
        ("Regulatory requirement", "Fixed Pipeline"),
        ("Unknown number of steps", "Dynamic Decomposition"),
        ("Exploratory / investigative", "Dynamic Decomposition"),
        ("Results change the plan", "Dynamic Decomposition"),
        ("Auditability required", "Fixed Pipeline"),
    ]
    for signal, strategy in signals:
        print(f"  {signal:<35} {strategy}")

visualize_strategy_comparison()

In [ ]:
#@title 🎧 Listen: Mini Project
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_mini_project.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Mini-Project — Hybrid Task Decomposition System

Build a system that automatically selects and executes the right decomposition strategy based on task characteristics.

In [ ]:
class HybridDecompositionSystem:
    """Automatically selects fixed or dynamic decomposition based on task analysis."""

    def __init__(self):
        self.execution_history = []

    def process_task(self, task: Dict, verbose: bool = True) -> Dict:
        """Analyze the task, choose a strategy, and execute it."""
        # Step 1: Choose strategy
        strategy = choose_strategy(task)

        if verbose:
            print(f"\n{'='*60}")
            print(f"TASK: {task['name']}")
            print(f"Strategy: {strategy['strategy']}")
            print(f"Rationale: {strategy['rationale']}")
            print(f"{'='*60}")

        # Step 2: Execute with chosen strategy
        if strategy["strategy"] == "fixed_pipeline":
            result = self._run_fixed(task, verbose)
        else:
            result = self._run_dynamic(task, verbose)

        # Step 3: Record result
        self.execution_history.append({
            "task": task["name"],
            "strategy": strategy["strategy"],
            "rationale": strategy["rationale"],
            "result": result,
        })

        return result

    def _run_fixed(self, task: Dict, verbose: bool) -> Dict:
        """Execute task using fixed pipeline."""
        steps = []
        for i, step_name in enumerate(task.get("steps", []), 1):
            steps.append(PipelineStep(
                name=step_name,
                prompt_template=f"Execute step: {step_name} for task {{task_name}}",
                required_inputs=["task_name"],
                output_key=f"step_{i}_result",
            ))

        pipeline = PromptChainPipeline(name=task["name"], steps=steps)
        result = pipeline.run({"task_name": task["name"]}, verbose=verbose)
        return {"method": "fixed_pipeline", "steps_completed": len(steps), "context": result}

    def _run_dynamic(self, task: Dict, verbose: bool) -> Dict:
        """Execute task using dynamic decomposition."""
        engine = DynamicDecompositionEngine()

        # Build target from task
        target = {
            "name": task["name"],
            "items": task.get("search_space", {
                "area_1": {"description": "Primary area", "relevance": 0.8,
                           "follow_ups": [f"investigate area_1 for {task['name']}"]},
                "area_2": {"description": "Secondary area", "relevance": 0.6,
                           "follow_ups": [f"investigate area_2 for {task['name']}"]},
                "area_3": {"description": "Tertiary area", "relevance": 0.3,
                           "follow_ups": [f"investigate area_3 for {task['name']}"]},
            })
        }

        engine.map_structure(target, verbose=verbose)
        plan = engine.assess_and_prioritize(threshold=0.5, verbose=verbose)
        results = engine.execute_plan(verbose=verbose)
        return {"method": "dynamic_decomposition", "steps_executed": engine.step_count,
                "findings": len(results)}

    def print_summary(self):
        """Print a summary of all processed tasks."""
        print(f"\n{'='*60}")
        print(f"HYBRID SYSTEM SUMMARY")
        print(f"{'='*60}")
        print(f"{'Task':<25} {'Strategy':<25} {'Steps'}")
        print("-" * 60)
        for entry in self.execution_history:
            steps = entry["result"].get("steps_completed",
                                         entry["result"].get("steps_executed", "?"))
            print(f"  {entry['task']:<23} {entry['strategy']:<25} {steps}")

# Run the hybrid system
system = HybridDecompositionSystem()

# Task 1: Fixed pipeline task
system.process_task({
    "name": "Loan Application",
    "steps": ["verify_identity", "pull_credit", "calculate_terms", "generate_offer"],
    "regulatory": True,
})

# Task 2: Dynamic task
system.process_task({
    "name": "Security Audit",
    "exploratory": True,
    "step_count": "unknown",
    "search_space": {
        "auth/": {"description": "Authentication module", "relevance": 0.95,
                  "follow_ups": ["audit auth/ for vulnerabilities"]},
        "api/": {"description": "API endpoints", "relevance": 0.8,
                 "follow_ups": ["audit api/ for injection risks"]},
        "config/": {"description": "Configuration files", "relevance": 0.7,
                    "follow_ups": ["audit config/ for exposed secrets"]},
        "static/": {"description": "Static assets", "relevance": 0.2,
                    "follow_ups": ["audit static/ for embedded scripts"]},
    },
})

# Task 3: Another fixed pipeline
system.process_task({
    "name": "Invoice Generation",
    "steps": ["gather_items", "calculate_totals", "apply_tax", "format_pdf", "send_email"],
    "regulatory": True,
})

system.print_summary()

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_15_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Key Takeaways

**Exam-critical concepts for Task Statement 1.6 (Task Decomposition):**

- **Fixed sequential pipelines (prompt chaining)** execute predetermined steps in order — predictable, debuggable, and ideal for well-defined workflows with regulatory requirements
- **Dynamic adaptive decomposition** lets the agent decide what to do next based on discoveries — flexible, model-driven, and essential for exploratory tasks with unknown scope
- **Per-file analysis + cross-file integration** is a two-phase pattern: analyze files independently first, then look for cross-cutting patterns that no single-file analysis could reveal
- **Map before planning** — always discover the territory before creating a prioritized plan. Do not jump straight into executing actions
- **Re-planning is adaptive** — when new information changes the picture, the agent should re-assess and re-prioritize, not blindly continue the original plan
- **Choose fixed pipelines** for known workflows, regulatory compliance, and auditability
- **Choose dynamic decomposition** for bug investigation, research, code review, and any task where the number of steps is unknown

In [ ]:
print("Notebook 4 complete!")
print()
print("Key exam decision framework:")
print("  Steps known + regulatory → Fixed Pipeline (prompt chaining)")
print("  Steps unknown + exploratory → Dynamic Adaptive Decomposition")
print("  Complex codebase → Per-file analysis + cross-file integration")
print()
print("Next: Session Management & Forking (NB 5)")